# Notebook 03: Equilibrium Analysis

## Negative Frequency-Dependent Selection (NFDS)

Under **self-incompatibility (SI)** at the S-locus, alleles experience **negative frequency-dependent selection**:

- **Rare alleles** confer a mating advantage — their carriers are compatible with a larger fraction of the population
- **Common alleles** are disadvantaged — more potential mates reject pollen carrying them

This is fundamentally different from classical Hardy-Weinberg equilibrium, which assumes no selection. Here, **selection is the driving force**: NFDS naturally pushes allele frequencies toward equality without any intervention.

### NFDS equilibrium target
The expected equilibrium under NFDS is **equal frequency of all S-alleles**:
- For `n` alleles: target frequency = `1/n` for each allele
- Real populations deviate from this due to genetic drift, bottlenecks, and founder effects
- In large undisturbed populations, NFDS alone will restore equilibrium over time
- In small or fragmented populations (like *L. papilliferum*), strategic crossing can **accelerate** what nature already does

### Distance metrics
We measure how far a population is from NFDS equilibrium using:
1. **Frequency variance**: `Var(frequencies)` — 0 at equilibrium
2. **Chi-squared statistic**: `Σ (observed - expected)² / expected`
3. **KL divergence**: `Σ p(x) log(p(x)/q(x))` where `q` is the uniform target

This notebook computes these metrics and simulates how crossing strategies affect allele frequency convergence.

In [1]:
import sys
import itertools
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Import shared utilities
sys.path.insert(0, "../src")
from polyploid_utils import (
    canonical, allele_frequencies, form_gametes, is_compatible,
    cross, crossing_compatibility,
)

print("Utilities loaded from polyploid_utils.")

Utilities loaded from polyploid_utils.


In [2]:
# Import equilibrium functions from shared module
from polyploid_utils import target_frequencies, distance_from_equilibrium

print("Equilibrium functions loaded: target_frequencies, distance_from_equilibrium")

Equilibrium functions loaded: target_frequencies, distance_from_equilibrium


In [3]:
# Import simulation and preservation functions from shared module
from polyploid_utils import (
    sample_offspring, simulate_generation,
    identify_rare_alleles, get_mandatory_rare_crosses, select_elites,
)


def compute_next_gen_frequencies(population, allele_pool, crossing_plan=None):
    """Compute expected allele frequencies after one generation of crossing."""
    next_gen = simulate_generation(population, crossing_plan=crossing_plan)
    return allele_frequencies(next_gen, allele_pool)


print("Simulation functions loaded: sample_offspring, simulate_generation")
print("Preservation helpers loaded: identify_rare_alleles, get_mandatory_rare_crosses, select_elites")

Simulation functions loaded: sample_offspring, simulate_generation
Preservation helpers loaded: identify_rare_alleles, get_mandatory_rare_crosses, select_elites


## Demo: Natural Convergence Under Random Mating

Starting from a **skewed population** (some alleles overrepresented), we simulate random compatible mating for several generations. Because of NFDS, rare alleles have a natural mating advantage — we expect allele frequencies to converge toward equality even without directed intervention. This demonstrates the self-correcting nature of frequency-dependent selection at the S-locus.

In [4]:
np.random.seed(42)
random.seed(42)

allele_pool = list(range(1, 9))  # 8 S-alleles

# Skewed starting population: alleles S1-S3 are overrepresented (common founders),
# while S7-S8 are rare (recently introduced or nearly lost to drift).
# This mimics a bottlenecked population of L. papilliferum where founder effects
# have created unequal allele frequencies — the starting point for our simulation.
population = [
    (1, 1, 2, 3),
    (1, 2, 2, 4),
    (1, 1, 3, 5),
    (2, 2, 3, 4),
    (1, 2, 5, 6),
    (1, 3, 4, 5),
    (2, 3, 3, 6),
    (1, 1, 2, 6),
    (1, 2, 4, 4),
    (2, 3, 5, 6),
    (1, 1, 4, 5),
    (1, 2, 3, 3),
    (2, 2, 5, 6),
    (1, 3, 4, 6),
    (1, 2, 2, 5),
    (1, 1, 3, 7),
    (2, 4, 5, 8),
    (1, 3, 6, 7),
    (2, 3, 4, 8),
    (1, 2, 7, 8),
]

print(f"Population size: {len(population)}")
print(f"Initial allele frequencies:")
initial_freqs = allele_frequencies(population, allele_pool)
for a, f in initial_freqs.items():
    print(f"  S{a}: {f:.4f}")

print(f"\nInitial distance from equilibrium:")
d = distance_from_equilibrium(population, allele_pool)
for metric, val in d.items():
    print(f"  {metric}: {val:.6f}")

Population size: 20
Initial allele frequencies:
  S1: 0.2375
  S2: 0.2250
  S3: 0.1625
  S4: 0.1125
  S5: 0.1000
  S6: 0.0875
  S7: 0.0375
  S8: 0.0375

Initial distance from equilibrium:
  variance: 0.005195
  chi_squared: 0.332500
  kl_divergence: 0.171652
  extinct_alleles: 0.000000
  endangered_alleles: 0.000000


In [5]:
# Simulate random mating over 30 generations to observe natural NFDS convergence.
# Each generation: the current population mates randomly, producing n_offspring
# individuals for the next generation. We track three distance metrics and
# per-allele frequencies at each generation to visualize the convergence trajectory.
n_generations = 30
pop_size = len(population)

# Track distance metrics and per-allele frequencies over generations
history = {
    "generation": [],
    "variance": [],
    "chi_squared": [],
    "kl_divergence": [],
}
freq_history = {a: [] for a in allele_pool}

current_pop = list(population)

for gen in range(n_generations + 1):
    # Record current state before advancing
    d = distance_from_equilibrium(current_pop, allele_pool)
    history["generation"].append(gen)
    history["variance"].append(d["variance"])
    history["chi_squared"].append(d["chi_squared"])
    history["kl_divergence"].append(d["kl_divergence"])

    freqs = allele_frequencies(current_pop, allele_pool)
    for a in allele_pool:
        freq_history[a].append(freqs[a])

    # Advance to next generation (except on the final iteration)
    if gen < n_generations:
        current_pop = simulate_generation(current_pop, n_offspring=pop_size)

print(f"Simulated {n_generations} generations of random mating.")
print(f"\nFinal distance metrics:")
for metric in ["variance", "chi_squared", "kl_divergence"]:
    print(f"  {metric}: {history[metric][0]:.6f} -> {history[metric][-1]:.6f}")

Simulated 30 generations of random mating.

Final distance metrics:
  variance: 0.005195 -> 0.000664
  chi_squared: 0.332500 -> 0.042500
  kl_divergence: 0.171652 -> 0.022461


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), layout="constrained")

# Plot 1: Frequency variance — overall measure of allele frequency imbalance.
# Decreasing variance means frequencies are becoming more uniform (converging).
axes[0].plot(history["generation"], history["variance"], "b-o", markersize=3)
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("Frequency variance")
axes[0].set_title("Frequency Variance Over Generations")
axes[0].axhline(y=0, color="red", linestyle="--", alpha=0.5, label="NFDS equilibrium")
axes[0].legend()

# Plot 2: Chi-squared — goodness-of-fit against uniform distribution.
# Approaching 0 means allele frequencies match the equilibrium target.
axes[1].plot(history["generation"], history["chi_squared"], "g-o", markersize=3)
axes[1].set_xlabel("Generation")
axes[1].set_ylabel("Chi-squared")
axes[1].set_title("Chi-Squared Distance Over Generations")
axes[1].axhline(y=0, color="red", linestyle="--", alpha=0.5, label="NFDS equilibrium")
axes[1].legend()

# Plot 3: Individual allele frequency trajectories.
# Under NFDS, rare alleles (low initial frequency) should increase and common
# alleles should decrease, all converging toward the 1/n target line.
target = 1.0 / len(allele_pool)
for a in allele_pool:
    axes[2].plot(history["generation"], freq_history[a], label=f"S{a}", linewidth=1.5)
axes[2].axhline(y=target, color="black", linestyle="--", alpha=0.5, label="NFDS target")
axes[2].set_xlabel("Generation")
axes[2].set_ylabel("Allele frequency")
axes[2].set_title("Allele Frequencies Over Generations\n(NFDS drives convergence to equal frequency)")
axes[2].legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)

plt.show()